# Week 3: The Deep Learning Balance - Practical Lab 🚀
Welcome to the practical Google Colab notebook for **Week 3: The Deep Learning Balance**. 

In the interactive lesson, you explored how to find the "sweet spot" for machine learning and deep learning models. Now, you will write the code to see these concepts in action!

### What we will cover:
1. **The Goldilocks Problem**: Underfitting vs. Overfitting.
2. **The Bias-Variance Tradeoff**: Visualizing the U-shaped error curve.
3. **Ensemble Methods**: Bagging (Random Forests) vs. Boosting.
4. **Regularization**: L1, L2, Dropout, and Early Stopping.
5. **Optimizers & Learning Rate**: Training neural networks with Adam and SGD.

In [ ]:
# Run this cell to import the required libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# For deep learning concepts, we'll use TensorFlow/Keras (Standard in Colab)
import tensorflow as tf
from tensorflow import keras

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Styling for plots
plt.style.use('dark_background')
print("Libraries imported successfully!")

---
## 1. The Goldilocks Problem (Underfitting vs. Overfitting)
Let's generate some noisy "True Pattern" data and try to fit it with models of varying complexity. 
- **Too Simple (Underfit)**: A straight line.
- **Just Right (Good Fit)**: A slight curve (e.g., a polynomial of degree 3).
- **Too Complex (Overfit)**: A wild, highly-wiggly curve that memorizes noise.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# 1. Generate a noisy sine wave
X = np.sort(np.random.rand(40) * 10)
y = np.sin(X) + np.random.randn(40) * 0.3  # True pattern is sin(x) + noise

X_plot = np.linspace(0, 10, 200)

# Create models of varying complexity
models = {
    "Too Simple (Underfit)": make_pipeline(PolynomialFeatures(1), LinearRegression()),
    "Just Right (Good Fit)": make_pipeline(PolynomialFeatures(3), LinearRegression()),
    "Too Complex (Overfit)": make_pipeline(PolynomialFeatures(15), LinearRegression())
}

plt.figure(figsize=(15, 4))
for i, (name, model) in enumerate(models.items()):
    ax = plt.subplot(1, 3, i + 1)
    model.fit(X[:, np.newaxis], y)
    y_plot = model.predict(X_plot[:, np.newaxis])
    
    ax.scatter(X, y, color='white', s=20, label="Training Data")
    ax.plot(X_plot, y_plot, color='#00D4AA', linewidth=2, label="Model Prediction")
    ax.set_title(name, fontsize=14)
    ax.set_ylim(-2, 2)
    if i == 0: ax.legend()

plt.tight_layout()
plt.show()

---
## 2. The Bias-Variance Tradeoff
The Total Error equation tells us: **Total Error = Bias² + Variance + Irreducible Noise**.
By testing models with increasing complexity (complexity = polynomial degree), we can visualize the U-shaped test error curve alongside the constantly decreasing training error.

In [ ]:
# Split the data to evaluate Train vs. Validation Loss
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42)

degrees = np.arange(1, 12)
train_errors, val_errors = [], []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train[:, np.newaxis], y_train)
    
    train_errors.append(mean_squared_error(y_train, model.predict(X_train[:, np.newaxis])))
    val_errors.append(mean_squared_error(y_val, model.predict(X_val[:, np.newaxis])))

# Plotting the Bias-Variance Tradeoff (Loss Curves)
plt.figure(figsize=(8, 5))
plt.plot(degrees, train_errors, label='Training Loss (Bias drops)', color='#4C9BE8', linewidth=2)
plt.plot(degrees, val_errors, label='Validation Loss (Variance rises eventually)', color='#F97316', linestyle='--', linewidth=2)
plt.xlabel('Model Complexity (Polynomial Degree)')
plt.ylabel('Mean Squared Error')
plt.title('Loss Curves: The U-Shaped Validation Error')
plt.ylim(0, 0.5)
plt.legend()
plt.show()

---
## 3. Ensemble Methods: Wisdom of the Crowd
As we learned, asking multiple models to vote usually yields better results.
- **Bagging (Random Forest)**: Trains many independent trees on random subsets to reduce variance.
- **Boosting (Gradient Boosting)**: Trains trees sequentially, where each tree fixes the mistakes of the previous one.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Let's create a more complex dataset
X_ens = np.linspace(0, 10, 100)
y_ens = np.sin(X_ens) + np.sin(3 * X_ens) + np.random.randn(100) * 0.3

# Initialize the three flavors of tree-based models
tree = DecisionTreeRegressor(max_depth=5)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42) # Bagging
xgb = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1) # Boosting

models = [("Single Decision Tree", tree), ("Random Forest (Bagging)", rf), ("Gradient Boosting", xgb)]

plt.figure(figsize=(15, 4))
for i, (name, model) in enumerate(models):
    model.fit(X_ens[:, np.newaxis], y_ens)
    y_pred = model.predict(X_plot[:, np.newaxis])
    
    ax = plt.subplot(1, 3, i + 1)
    ax.scatter(X_ens, y_ens, color='white', s=10, alpha=0.5)
    ax.plot(X_plot, y_pred, color='#F59E0B', linewidth=2)
    ax.set_title(name)

plt.tight_layout()
plt.show()

---
## 4. Regularization in Neural Networks
Now we switch to Deep Learning! We will deliberately create an overpowered network so it **overfits**. Then we will apply different regularization techniques:
1. **Unregularized**: Memorizes the data (Overfits).
2. **Dropout**: Randomly deactivates neurons so the network can't rely on just a few paths.
3. **L2 (Weight Decay)**: Shrinks weights to stop them from becoming too large.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2

# Generate a moon dataset (classic binary classification)
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=42)
X_m_train, X_m_val, y_m_train, y_m_val = train_test_split(X_moons, y_moons, test_size=0.5, random_state=42)

def build_model(use_dropout=False, use_l2=False):
    model = Sequential()
    # Large hidden layers to purposefully encourage overfitting
    model.add(Dense(128, activation='relu', input_shape=(2,), 
                    kernel_regularizer=l2(0.01) if use_l2 else None))
    if use_dropout: model.add(Dropout(0.4))  # Drop 40% of neurons
    
    model.add(Dense(128, activation='relu',
                    kernel_regularizer=l2(0.01) if use_l2 else None))
    if use_dropout: model.add(Dropout(0.4))
        
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train the Overfitting (Unregularized) Model
print("1. Training Unregularized Model...")
model_base = build_model()
hist_base = model_base.fit(X_m_train, y_m_train, epochs=200, validation_data=(X_m_val, y_m_val), verbose=0)

# Train the Dropout Model
print("2. Training Dropout Model...")
model_drop = build_model(use_dropout=True)
hist_drop = model_drop.fit(X_m_train, y_m_train, epochs=200, validation_data=(X_m_val, y_m_val), verbose=0)

# Identify the gap between train and validation loss
plt.figure(figsize=(12, 5))
plt.plot(hist_base.history['loss'], label='Base Train Loss', color='#4C9BE8')
plt.plot(hist_base.history['val_loss'], label='Base Val Loss (Overfitting!)', color='#F97316', linestyle='--')
plt.plot(hist_drop.history['loss'], label='Dropout Train Loss', color='#00D4AA')
plt.plot(hist_drop.history['val_loss'], label='Dropout Val Loss (Stable)', color='#10B981', linestyle='--')
plt.title("Effect of Dropout Regularization on Overfitting")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

---
## 5. Optimizers and Learning Rate
The **Learning Rate** is the most important hyperparameter. 
- **Too high**: The loss oscillates and explodes (fails to find the valley).
- **Too low**: It takes forever to train.
- **Optimizers**: Adam automatically adapts the learning rate, while SGD requires careful tuning.

In [ ]:
def train_with_lr(lr, optimizer_name='sgd'):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(2,)),
        Dense(1, activation='sigmoid')
    ])
    
    if optimizer_name == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=lr)
    else:
        opt = keras.optimizers.Adam(learning_rate=lr)
        
    model.compile(optimizer=opt, loss='binary_crossentropy')
    # Train for just 50 epochs to see immediate learning rate effects
    h = model.fit(X_m_train, y_m_train, epochs=50, validation_data=(X_m_val, y_m_val), verbose=0)
    return h.history['loss']

print("Training with different learning rates...")

# 1. Good Learning Rate
loss_good = train_with_lr(0.01, 'adam')
# 2. Too Low Learning Rate
loss_low = train_with_lr(0.0001, 'sgd')
# 3. Too High Learning Rate
loss_high = train_with_lr(2.0, 'sgd')

plt.figure(figsize=(10, 5))
plt.plot(loss_good, label='Good LR (0.01, Adam) - Smooth Convergence', color='#00D4AA', linewidth=2)
plt.plot(loss_low, label='Too Low LR (0.0001, SGD) - Barely Moving', color='#F97316', linewidth=2)
plt.plot(loss_high, label='Too High LR (2.0, SGD) - Chaotic/Oscillating', color='#EF4444', linewidth=2)
plt.ylim(0, 1.5)
plt.title("Learning Rate Behavior")
plt.xlabel("Epochs")
plt.ylabel("Training Loss")
plt.legend()
plt.show()

### Congratulations! 🎉
You have just put into practice all the concepts from the **Deep Learning Balance** lesson. By touching real code, you’ve seen firsthand how tweaking regularizers, capacity, and learning rates affects how a model generalizes. 

*You are no longer just a user. You are a builder.*